# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshithavandavasi-1401/HV-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
print(os.getcwd())

/content


In [5]:
!git clone https://github.com/harshithavandavasi-1401/HV-flyrank-ml-internship.git

Cloning into 'HV-flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 48), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 5.51 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [6]:
%cd /content/HV-flyrank-ml-internship

/content/HV-flyrank-ml-internship


In [7]:
import os
print(os.getcwd())

/content/HV-flyrank-ml-internship


## 1. Build the feature vector

### Answer

I built the feature vector using historical SEO, traffic, engagement, and content-related features that are available before prediction. Numerical missing values are filled with the median, categorical missing values are filled with "Unknown", and categorical features are one-hot encoded. The target column (`trend_direction`) is not included in the feature vector.

In [10]:
# Features selected for the model
feature_columns = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "content_type",
    "main_intent"
]

# Build feature vector
X = df[feature_columns].copy()

# Fill missing values
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna("Unknown")

# One-hot encode categorical features
X = pd.get_dummies(X, columns=cat_cols)

print("Feature vector shape:", X.shape)
print("\nSample columns:")
print(X.columns.tolist()[:15])

Feature vector shape: (30000, 18)

Sample columns:
['search_volume', 'word_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_Unknown', 'main_intent_commercial']


## 2. Feature notes (meaning, missing, categorical, available-when?)

### Answer

| Feature | Meaning | Missing Value Handling | Available Before Prediction? |
|---------|---------|-----------------------|------------------------------|
| search_volume | Keyword search demand | Filled with median | Yes |
| word_count | Number of words in content | Filled with median | Yes |
| impressions_90d | Impressions in last 90 days | No missing values | Yes |
| clicks_90d | Clicks in last 90 days | No missing values | Yes |
| sessions_90d | Sessions in last 90 days | No missing values | Yes |
| ctr | Click-through rate | No missing values | Yes |
| avg_position | Average search ranking | No missing values | Yes |
| engagement_rate | User engagement rate | No missing values | Yes |
| content_age_days | Age of the content | No missing values | Yes |
| days_since_last_update | Days since last update | No missing values | Yes |
| content_type | Type of content | Filled with "Unknown" if needed and one-hot encoded | Yes |
| main_intent | Search intent | Filled with "Unknown" if needed and one-hot encoded | Yes |

In [11]:
notes = pd.DataFrame({
    "Feature": feature_columns,
    "Missing Values": df[feature_columns].isnull().sum().values,
    "Data Type": df[feature_columns].dtypes.values
})

print(notes)

                   Feature  Missing Values Data Type
0            search_volume            2468   float64
1               word_count            7699   float64
2          impressions_90d               0     int64
3               clicks_90d               0     int64
4             sessions_90d               0     int64
5                      ctr               0   float64
6             avg_position               0   float64
7          engagement_rate               0   float64
8         content_age_days               0     int64
9   days_since_last_update               0     int64
10            content_type               0    object
11             main_intent            2374    object


## 3. The leakage hunt

### Answer

I checked the selected features for possible data leakage.

The feature `trend_pct` was identified as a leakage feature because it directly represents the future trend used to determine the prediction target (`trend_direction`). Including this feature would allow the model to indirectly see the answer.

The remaining selected features are historical measurements collected before prediction, so they are appropriate for training.

In [12]:
possible_leakage = [
    "trend_pct",
    "trend_direction"
]

print("Potential Leakage Columns:")
for col in possible_leakage:
    print("-", col)

print("\nColumns excluded from training:")
print(possible_leakage)

Potential Leakage Columns:
- trend_pct
- trend_direction

Columns excluded from training:
['trend_pct', 'trend_direction']


## 4. What I excluded and why

### Answer

| Column | Reason for Exclusion |
|---------|---------------------|
| trend_direction | Prediction target |
| trend_pct | Direct target leakage |
| content_id | Unique identifier with no predictive meaning |
| client_id | Identifier that may cause memorization instead of learning patterns |

In [14]:
excluded_features = pd.DataFrame({
    "Column":[
        "trend_direction",
        "trend_pct",
        "content_id",
        "client_id"
    ],
    "Reason":[
        "Target variable",
        "Target leakage",
        "Unique identifier",
        "Unique identifier"
    ]
})

excluded_features

,Column,Reason
0,trend_direction,Target variable
1,trend_pct,Target leakage
2,content_id,Unique identifier
3,client_id,Unique identifier


## Self-check

- ✅ Every section is completed with both markdown and code.
- ✅ The notebook runs from top to bottom without errors.
- ✅ No client names, URLs, or private information are included.
- ✅ The notebook clearly distinguishes historical features from leakage features.
- ✅ The notebook is ready to commit to the repository.